# Setup

In [1]:
import os, sys

SUMO_HOME  = 'C:\Program Files (x86)\Eclipse\Sumo'
# PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'

os.environ['SUMO_HOME']  = SUMO_HOME
# os.environ['PROJ_LIB']   = PROJ_PATH
# os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
#     'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ✅
gtfs2pt.py     : ✅


<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Soheil99\AppData\Local\Temp\ipykernel_42224\516504717.py:3: SyntaxWarning: invalid escape sequence '\P'
  SUMO_HOME  = 'C:\Program Files (x86)\Eclipse\Sumo'


In [2]:
# cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2
!cd "C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS"
!cd

C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS


# OSM Map Extraction

-extraction from Geofrabil for Full Washington Link Network

In [24]:
import requests

# Download Washington state extract from Geofabrik
url = "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf"
out = "washington.osm.pbf"

print("Downloading Washington state from Geofabrik (~100MB)...")
with requests.get(url, stream=True) as r:
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(out, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f} MB)", end='')

print(f"\n✅ Done — saved to {out}")

  100.0% (339.4 MB)
✅ Done — saved to washington.osm.pbf


In [52]:
!osmium extract \
  -b -122.4500,47.2500,-122.1000,47.8500 \
  washington.osm.pbf \
  -o seattle_link.osm.pbf \
  --strategy complete_ways

No extract specified in config file or on the command line.


In [56]:
!osmium extract \
  -b -124.8,45.5,-116.9,49.1 \
  washington.osm.pbf \
  -o washington_full.osm.pbf \
  --strategy complete_ways

[======================================================================] 100% 


In [36]:
!osmium cat seattle_link.osm.pbf -o seattle_link.osm.xml

[======================================================================] 100% 


In [58]:
!osmium cat washington_full.osm.pbf -o seattle_link_full.osm.xml

[======================================================================] 100% 


# Network Preparation

In [ ]:
!netconvert \
  --osm-files seattle_link.osm.xml \
  -o seattle_lightrail_3.net.xml \
  --type-files $SUMO_HOME/data/typemap/osmNetconvert.typ.xml,$SUMO_HOME/data/typemap/osmNetconvertRailUsage.typ.xml \
  --keep-edges.by-type railway.light_rail,railway.subway \
  --proj.utm true \
  --geometry.remove \
  --junctions.join \
  --output.street-names \
  --ptstop-output seattle_rail_stops.add.xml \
  --ptline-output seattle_rail_ptlines.add.xml \
  --osm.stop-output.length 30

# GTFS

In [7]:
import pandas as pd
import zipfile
import os

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data


### Filtering to major Seattle GTFS Bus Lines

In [8]:

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/kcm_google_transit.zip"
OUTPUT_ZIP = "gtfs data/kcm_google_transit_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = ["24", "33", "70", "8"]

# KEEP_ROUTES = [
#     "1", "2", "3", "4", "7", "8", "40",
#     "101", "120", "C Line", "D Line", "E Line", "H Line"
# ]

# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.58, 47.65
LON_MIN, LON_MAX = -122.37, -122.30

# ==============================
# LOAD GTFS FILES
# ==============================



gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]
## TODO ---soheil comment:
# do we have trips where the removed stops are in mid trips? trips that leave simulation area and then come back to it?
print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist --- soheil: Not sure why not ALL unchanged files are not copied into the new zip
    for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
        if fname in gtfs:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 4
Kept trips: 1270
Kept stops in downtown: 649
✅ Filtered GTFS saved to: gtfs data/kcm_google_transit_downtown.zip


In [13]:
# busgtfs = load_gtfs('gtfs data/kcm_google_transit_downtown.zip')
# stops = busgtfs['stops.txt']
# stops.stop_name.unique()
# stops.head()

In [14]:
# railgtfs = load_gtfs('gtfs data/rail_gtfs.zip')
# stops = railgtfs['stops.txt']
# # stops.stop_name.unique()
# # stops.head()
# # stops[stops["stop_name"].str.contains("Lynnwood", case=False, na=False)]
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126


# filtered_stops = stops[
#     (stops["stop_lat"] >= LAT_MIN) &
#     (stops["stop_lat"] <= LAT_MAX) &
#     (stops["stop_lon"] >= LON_MIN) &
#     (stops["stop_lon"] <= LON_MAX)
# ]

# filtered_stops

In [28]:
# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/rail_gtfs.zip"
OUTPUT_ZIP = "gtfs data/rail_gtfs_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1 Line", "2 Line"
]

# Downtown Seattle bounding box
# #old ones
LAT_MIN, LAT_MAX = 47.31, 47.84
LON_MIN, LON_MAX = -122.43, -122.09

# new ones
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126
# 47.5798124250899, -122.32752854299258 (below SODO)
# 47.65143948560587, -122.30397070065698 (above UW)
# 47.59038693450619, -122.29961918065126 (judkins park)
# 47.59525640734678, -122.38544987028668 (west seattle)

# ==============================
# LOAD GTFS FILES
# ==============================

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
#     for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
#         if fname in gtfs:
#             write(gtfs[fname], fname)
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 2
Kept trips: 17803
Kept stops in downtown: 309
✅ Filtered GTFS saved to: gtfs data/rail_gtfs_downtown.zip


### Converging new Lightrail Segments

In [ ]:
!netconvert -s soheil_seattle.net.xml --remove-edges.by-vclass rail,rail_urban,rail_fast,subway,rail_electric -o soheil_seattle_veh.net.xml

In [ ]:
!netconvert -s seattle_lightrail_2.net.xml,soheil_seattle_veh.net.xml -o merged.net.xml 

#### Adding Full Lightrail Network

In [5]:
!netconvert -s "Zili\rail\toy network\seattle_lightrail_DT_excluded.net.xml",soheil_seattle_correct_premissions.net.xml  -o soheil_seattle_merged_v2.net.xml 

Success.


In [5]:
!netconvert -s soheil_seattle_correct_premissions.net.xml,"Zili\rail\toy network\seattle_lightrail_DT_excluded.net.xml" -o soheil_seattle_merged.net.xml 

Success.


### Adding GTFS to Sumo Network

#### suggestion: make sure previous outputs are deleted (fcd and resources folders)

In [6]:
# !python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
# -n merged.net.xml \
# --gtfs rail_gtfs_fil.zip \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

# !python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
# -n merged_v2.net.xml \
# --gtfs "gtfs data/rail_gtfs_downtown.zip" \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose


!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged.net.xml \
--gtfs "gtfs data/rail_gtfs_downtown.zip" \
--date 20260329 \
--modes=tram \
--repair \
--verbose

Loading net
function import_gtfs called at Tue, 23 Jun 2026 12:39:29 +0000
Loading GTFS data "gtfs data/rail_gtfs_downtown.zip"
function import_gtfs finished after 0.711240 seconds
Success.
Success.
Writing fcd file "fcd\gtfs\tram.fcd.xml"
mapping tram
mapping trace with 15 points ... (584 router calls)
mapping trace with 26 points ... (655 router calls)
mapping trace with 22 points ... (616 router calls)
mapping trace with 26 points ... (655 router calls)
mapping trace with 4 points ... (9 router calls)
mapping trace with 8 points ... (21 router calls)
mapping trace with 11 points ... (49 router calls)
mapping trace with 22 points ... (535 router calls)
mapping trace with 11 points ... (49 router calls)
mapping trace with 26 points ... (539 router calls)
mapping trace with 26 points ... (539 router calls)
mapping trace with 1 points ... (0 router calls)
mapping trace with 4 points ... (31 router calls)
mapping trace with 6 points ... (33 router calls)
mapping trace with 10 points ... 

Warning! No mapping library found, falling back to tracemapper.


In [9]:
!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged.net.xml \
--gtfs "gtfs data/kcm_google_transit_downtown.zip" \
--date 20260617 \
--modes=bus \
--repair \
--verbose

Loading net
function import_gtfs called at Tue, 23 Jun 2026 15:43:33 +0000
Loading GTFS data "gtfs data/kcm_google_transit_downtown.zip"
function import_gtfs finished after 0.070888 seconds
Success.
Success.
Writing fcd file "fcd\gtfs\bus.fcd.xml"
mapping bus
mapping trace with 6 points ... (4632 router calls)
mapping trace with 16 points ... (17143 router calls)
mapping trace with 12 points ... (9742 router calls)
mapping trace with 11 points ... (10427 router calls)
mapping trace with 16 points ...
   Found no candidate edges for 4149.59,8772.55 (index 0)
   Found no candidate edges for 3983.07,8021.06 (index 1)
   Found no candidate edges for 3993.40,7430.11 (index 2)
   Found no candidate edges for 3430.21,6336.43 (index 4)
4 Points had no candidates. (12223 router calls)
mapping trace with 13 points ...
   Found no candidate edges for 4019.53,7387.51 (index 10)
   Found no candidate edges for 3998.15,8067.41 (index 11)
   Found no candidate edges for 4185.39,8809.75 (index 12)
3 P

Warning! No mapping library found, falling back to tracemapper.
Trip 801592550 (bus): detour (factor 10.85) to stop index 8, fromPos=3382.86,5838.69 toPos=3437.25,6321.23 (airLine=485.60 path=5270.28)
Trip 801592550 (bus): detour (factor 12.69) to stop index 9, fromPos=3437.25,6321.23 toPos=3754.46,6697.50 (airLine=492.14 path=6243.46)
Trip 801592840 (bus): detour (factor 10.85) to stop index 9, fromPos=3382.86,5838.69 toPos=3437.25,6321.23 (airLine=485.60 path=5270.28)
Trip 801592840 (bus): detour (factor 12.69) to stop index 10, fromPos=3437.25,6321.23 toPos=3754.46,6697.50 (airLine=492.14 path=6243.46)
Trip 800890320 (bus): detour (factor 26.79) to stop index 20, fromPos=1798.17,5919.43 toPos=1703.37,6088.23 (airLine=193.60 path=5186.93)
Trip 800890360.trimmed (bus): detour (factor 26.79) to stop index 2, fromPos=1798.17,5919.43 toPos=1703.37,6088.23 (airLine=193.60 path=5186.93)
Warning! Disconnected route '801592540' between '4755256' and '-393047815#1', no path found. Keeping lon

In [ ]:

with open("tram.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="tram"
           vClass="rail"
           accel="1.2"
           decel="1.5"
           sigma="0.3"
           length="40"
           maxSpeed="25"
           guiShape="rail"/>
</additional>
""")


In [6]:
with open("bus.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="bus"
           vClass="bus"
           accel="1.0"
           decel="4.5"
           sigma="0.5"
           length="12"
           maxSpeed="16.7"
           guiShape="bus"/>
</additional>
""")


In [12]:
import re

with open("gtfs_pt_stops.add.xml") as f:
    content = f.read()

# replace trainStop with busStop
content = content.replace("trainStop", "busStop")

with open("gtfs_pt_stops_fixed.add.xml", "w") as f:
    f.write(content)


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml